In [4]:
import sys
import importlib
from pathlib import Path

import joblib
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.data.preprocess import notebook_root
from src.models.data import load_baseline_inputs

import src.models.xgboost_model as xgb_model

importlib.reload(xgb_model)

root = notebook_root()
inputs = load_baseline_inputs(root)

print("Data loaded successfully.")
import importlib
import src.models.xgboost_model as xgb_model

importlib.reload(xgb_model)

print(xgb_model.f1_score)

Data loaded successfully.
<function f1_score at 0x0000026F9750D4E0>


In [5]:
tuning_results, best_models = xgb_model.tune_xgboost(inputs)

results_dir = root / "results" / "xgboost"
results_dir.mkdir(parents=True, exist_ok=True)

tuning_results.to_csv(
    results_dir / "tuning_results.csv",
    index=False
)

for feature_set, model in best_models.items():
    joblib.dump(
        model,
        results_dir / f"{feature_set}_tuned.joblib"
    )

print("Tuning complete! Results and models saved.")

Training core, configuration 1/10...
Training core, configuration 2/10...
Training core, configuration 3/10...
Training core, configuration 4/10...
Training core, configuration 5/10...
Training core, configuration 6/10...
Training core, configuration 7/10...
Training core, configuration 8/10...
Training core, configuration 9/10...
Training core, configuration 10/10...
Training expanded, configuration 1/10...
Training expanded, configuration 2/10...
Training expanded, configuration 3/10...
Training expanded, configuration 4/10...
Training expanded, configuration 5/10...
Training expanded, configuration 6/10...
Training expanded, configuration 7/10...
Training expanded, configuration 8/10...
Training expanded, configuration 9/10...
Training expanded, configuration 10/10...
Tuning complete! Results and models saved.


In [6]:
# Select the best configuration for each feature set
best_results = (
    tuning_results
    .sort_values("macro_f1", ascending=False)
    .drop_duplicates(subset="feature_set")
    .reset_index(drop=True)
)

display(best_results)

,feature_set,configuration,n_estimators,max_depth,learning_rate,subsample,accuracy,macro_f1,high_severity_recall
0,expanded,3,100,9,0.1,1.0,0.651722,0.352255,0.054642
1,core,3,100,9,0.1,1.0,0.647120,0.277727,0.014189


In [7]:
winner = best_results.iloc[0]

print("BEST XGBOOST MODEL")
print("------------------")
print("Feature set:", winner["feature_set"])
print("Configuration:", winner["configuration"])
print("Macro-F1:", round(winner["macro_f1"], 4))
print("Accuracy:", round(winner["accuracy"], 4))
print("High-severity recall:", round(winner["high_severity_recall"], 4))

print("\nHyperparameters:")
for param in ["n_estimators", "max_depth", "learning_rate", "subsample"]:
    print(f"{param}: {winner[param]}")

BEST XGBOOST MODEL
------------------
Feature set: expanded
Configuration: 3
Macro-F1: 0.3523
Accuracy: 0.6517
High-severity recall: 0.0546

Hyperparameters:
n_estimators: 100
max_depth: 9
learning_rate: 0.1
subsample: 1.0
